# Flowmetrics fulfillment analysis

DuckDB SQL and pandas analysis for the fulfillment case study.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
db = duckdb.connect(str(ROOT / 'data/processed/flowmetrics.duckdb'))
orders = db.sql('SELECT * FROM orders').df()
events = db.sql('SELECT * FROM fulfillment_events').df()
orders.head()

## Cleaning and validation

Parse timestamps, verify order grain and chronology, then derive the SLA flag and order hour.

In [ ]:
for column in ['order_ts', 'ship_ts', 'delivery_ts', 'promised_delivery_ts']:
    orders[column] = pd.to_datetime(orders[column], utc=True)
assert orders['order_id'].is_unique
assert orders[['order_ts', 'ship_ts', 'delivery_ts']].notna().all().all()
assert (orders['delivery_ts'] >= orders['order_ts']).all()
orders['sla_breach'] = (orders['delivery_ts'] > orders['promised_delivery_ts']).astype(int)
orders['order_hour'] = orders['order_ts'].dt.hour
orders.describe(include='all').T

## Business questions

In [ ]:
stage_query = '''
SELECT event_type, round(avg(datediff('minute', order_ts, event_ts) / 60.0), 2) AS avg_hours
FROM fulfillment_events e JOIN orders o USING (order_id)
GROUP BY event_type ORDER BY avg_hours
'''
db.sql(stage_query).df()

In [ ]:
site_sla = (orders.groupby('site_id', as_index=False)
    .agg(orders=('order_id', 'size'), avg_process_hours=('process_hours', 'mean'), sla_breach_rate=('sla_breach', 'mean'))
    .sort_values('sla_breach_rate', ascending=False))
site_sla['sla_breach_rate'] = site_sla['sla_breach_rate'].mul(100).round(1)
site_sla

In [ ]:
driver_summary = (orders.groupby(['priority', 'channel'], as_index=False)
    .agg(orders=('order_id', 'size'), avg_process_hours=('process_hours', 'mean'), sla_breach_rate=('sla_breach', 'mean')))
driver_summary['sla_breach_rate'] = driver_summary['sla_breach_rate'].mul(100).round(1)
driver_summary.sort_values('avg_process_hours', ascending=False).head(10)

In [ ]:
hourly = orders.groupby('order_hour', as_index=False).agg(orders=('order_id', 'size'), avg_process_hours=('process_hours', 'mean'))
sns.lineplot(data=hourly, x='order_hour', y='orders', marker='o')
plt.title('Order volume by hour of day')
plt.tight_layout()
plt.show()
hourly